In [ ]:
import serial
import time
import cv2
import numpy as np
import tensorflow as tf
import joblib

In [ ]:
arduino = serial.Serial(port='COM3', baudrate=9600, timeout=0.1)
keras_model = tf.keras.models.load_model('resnet_hybrid_keras_model.keras')
xgb_model = joblib.load('xgboost_classifier_model.joblib')
scaler = joblib.load('scaler.joblib') # Save your scaler as a file!

In [ ]:
# Extract feature layer
feature_extractor = tf.keras.Model(
    inputs=keras_model.input, 
    outputs=keras_model.get_layer("feature_extraction_layer").output
)

cap = cv2.VideoCapture(0)
classes = ['defects', 'good weld']

def run_hybrid_prediction():
    ret, frame = cap.read()
    if not ret: return None
    
    # Preprocess (matches your 300x300 requirement)
    img = cv2.resize(frame, (300, 300))
    img_array = img.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_array, axis=0)

    # Hybrid Prediction Pipeline
    features = feature_extractor.predict(img_batch, verbose=0)
    flat_features = features.reshape(1, -1)
    scaled_features = scaler.transform(flat_features)
    
    prediction_idx = int(xgb_model.predict(scaled_features)[0])
    return classes[prediction_idx]

print("System Active. Waiting for Arduino trigger...")


In [ ]:
while True:
    # Listen for "TRIGGER" from Arduino sensor
    if arduino.in_waiting > 0:
        line = arduino.readline().decode('utf-8').strip()
        
        if line == "TRIGGER":
            print("Part detected. Capturing image...")
            result = run_hybrid_prediction()
            print(f"Result: {result}")
            
            # Send 'D' for defect, 'G' for good weld back to Arduino
            msg = 'D' if result == 'defects' else 'G'
            arduino.write(msg.encode())